In [2]:
!pip install emoji requests spacy scikit-learn pandas
!python -m spacy download pl_core_news_sm

import re
import emoji
import string
import requests
import spacy
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

nlp = spacy.load("pl_core_news_sm")


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
Traceback (most recent call last):
  File "<frozen runpy>", line 189, in _run_module_as_main
  File "<frozen runpy>", line 148, in _get_module_details
  File "<frozen runpy>", line 112, in _get_module_details
  File "E:\Programming\przetwarzanie\.venv\Lib\site-packages\spacy\__init__.py", line 18, in <module>
    from .cli.info import info  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "E:\Programming\przetwarzanie\.venv\Lib\site-packages\spacy\cli\__init__.py", line 4, in <module>
    from . import download as download_module  # noqa: F401
    ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "E:\Programming\przetwarzanie\.venv\Lib\site-packages\spacy\cli\download.py", line 20, in <module>
    from ._util import SDIST_SUFFIX, WHEEL_SUFFIX, Arg, Opt, app
  File "E:\Programming\przetwarzanie\.venv\Lib\site-packages\spacy\cli\_util.py", line 18, in <module>
 

ModuleNotFoundError: No module named 'click'

#Zadania - część 1
## tekst do przetworzenia w ćwiczeniach

taka_historia = '<p> Cześć!!! 👋 Witajcie na moim nowym blogu o sztucznej inteligencji...   Dzisiaj porozmawiamy o NLP (Natural Language Processing). </p>

Czy wiedzieliście, że aż 80% danych w firmach to dane nieustrukturyzowane??? 😲 Więcej informacji znajdziecie na stronie: https://www.przykladowastrona.pl/nlp-wstep lub pisząc na e-mail: kontakt@moj-blog-ai.com.pl.

#MachineLearning #DataScience @Kowalski_Data_Geek

W      niektórych    miejscach celowo zostawiłem  duuuuuużo spacji i tabulacji. ALBO NAPISAŁEM COŚ CAPSLOCKIEM, żebyście mieli co zmieniać na małe litery (tzw. lowercasing).
Warto usunąć z tego tekstu polskie stop-words, np.: "i", "w", "na", "oraz", "że".

Data publikacji: 12.05.2023 r., godz. 14:30. Zysk firmy wzrósł o $45,000 w Q3!
P.S. Nie zapomnijcie o usunięciu tagów HTML, np. <b>pogrubienia</b> i znaków interpunkcyjnych! 🚀'

### Zadanie 1
a) zamień wielkość znaków na znaki małe
b) pozbądź się nadmiarowych białych znaków
c) pozbądź się nadmiarowych znaków przestankowych następujących po sobie (!!!, ???). Możesz je znaleźć w module string.
d) zamień emotikony na ich tekstową postać
e) pozbądź się wszystkich tagów HTML

### Zadanie 2
Masz do dyspozycji trzy tagi:
- <NUM> - liczby, daty, wartości księgowe i wszystko co stanowi wartość numeryczną,
- <URL> - adresy URL
- <EMAIL> - adresy email
Za pomocą wyrażeń regularnych i/lub funkcji wbudowanych klasy str zamień wartości w tekście na powyższe tagi.

### Zadanie 3
Wykorzystując listę stop words z adresu https://github.com/bieli/stopwords/blob/master/polish.stopwords.txt pozbądź się wszystkich słów stop z tekstu.

In [3]:
# Przykładowy, skrajnie "brudny" tekst do przetestowania naszego potoku
brudny_tekst = """ '<p> Cześć!!! 👋 Witajcie na moim nowym blogu o sztucznej inteligencji...   Dzisiaj porozmawiamy o NLP (Natural Language Processing). </p>
"""

# --- ZADANIE 1 ---
def oczysc_tekst_zad1(tekst):
    tekst = tekst.lower() # a)
    tekst = re.sub(r'<[^>]+>', '', tekst) # e)
    tekst = emoji.demojize(tekst) # d)

    # c)
    for znak in string.punctuation:
        znak_escaped = re.escape(znak)
        tekst = re.sub(f'({znak_escaped}){{2,}}', r'\1', tekst)

    tekst = re.sub(r'\s+', ' ', tekst).strip() # b)
    return tekst

# --- ZADANIE 2 ---
def taguj_tekst_zad2(tekst):
    tekst = re.sub(r'http[s]?://(?:[a-zA-Z]|[0-9]|[$-_@.&+]|[!*\(\),]|(?:%[0-9a-fA-F][0-9a-fA-F]))+', '<URL>', tekst)
    tekst = re.sub(r'[a-zA-Z0-9_.+-]+@[a-zA-Z0-9-]+\.[a-zA-Z0-9-.]+', '<EMAIL>', tekst)
    tekst = re.sub(r'\b\d+\b', '<NUM>', tekst)
    return tekst

# --- ZADANIE 3 ---
url_stop = "https://raw.githubusercontent.com/bieli/stopwords/master/polish.stopwords.txt"
stop_words_pl = set(requests.get(url_stop).text.splitlines())

def usun_stopwords_zad3(tekst, stopwords):
    slowa = tekst.split()
    slowa_przefiltrowane = [slowo for slowo in slowa if slowo not in stopwords]
    return " ".join(slowa_przefiltrowane)


zadanie1 = oczysc_tekst_zad1(brudny_tekst)
zadanie2 = taguj_tekst_zad2(krok1)
zadanie3 = usun_stopwords_zad3(krok2, stop_words_pl)

print("Zadanie 1 (Podstawowe czyszczenie):", zadanie1)
print("Zadanie 2 (Tagowanie Regex):", zadanie2)
print("Zadanie 3 (Usunięcie StopWords):", zadanie3)

KROK 1 (Podstawowe czyszczenie): ' cześć! :waving_hand: witajcie na moim nowym blogu o sztucznej inteligencji. dzisiaj porozmawiamy o nlp (natural language processing).
KROK 2 (Tagowanie Regex): ' cześć! :waving_hand: witajcie na moim nowym blogu o sztucznej inteligencji. dzisiaj porozmawiamy o nlp (natural language processing).
KROK 3 (Usunięcie StopWords): ' cześć! :waving_hand: witajcie nowym blogu sztucznej inteligencji. porozmawiamy nlp (natural language processing).
